# 🚚 F0 — Setup DEMO (instalar datos de prueba en `data/00_raw/`)

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la Universitat Jaume I**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Versión** | AU_UJI Dinámico (V2) |
| **Fase** | 0 — Configuración |
| **Tipo** | Notebook auxiliar (uso opcional) |

---

## 🎯 ¿Qué hace este notebook?

Descomprime el fichero `data/demo/demo_excel.zip` (que contiene los 2 Excel reducidos con 50 alumnos ficticios) y los copia a `data/00_raw/` con los **nombres exactos** que esperan los notebooks del pipeline:

- `datos_proyecto_sin_preinscrip.xlsx`
- `preinscripcion_si.xlsx`

Después de ejecutar este notebook, las fases F1 → F7 funcionan **sin tocar nada más**.

## 🛡️ Protecciones automáticas

El notebook **NO sobrescribe nunca tus datos reales**:

1. Si en `data/00_raw/` ya hay ficheros con esos nombres, comprueba el **número de filas** del Excel principal:
    - Si tiene **más de 1.000 filas** → considera que son tus datos reales (no DEMO) y **NO hace nada**, avisa por pantalla.
    - Si tiene **menos de 100 filas** → asume que son DEMO antiguos y los **renombra como `_BACKUP_<fecha_hora>.xlsx`** antes de copiar los nuevos.
2. Si encuentra una carpeta antigua `data/00_raw/ejemplo/` (de versiones anteriores), la **detecta y avisa** pero NO la borra.
3. El fichero `datos_proyecto_sin_preinscrip.ORIGINALxlsx` (con extensión protegida que indica el dataset original con todos los alumnos de la Comunitat Valenciana) **NUNCA se toca**.

## 🧭 Cuándo ejecutar este notebook

**Sí ejecutarlo:**
- Si has descargado el repo de GitHub y quieres probar el pipeline con los 50 alumnos de prueba.

**NO ejecutarlo:**
- Si has recibido los Excel originales (de María José vía Drive o de la UJI) y los has puesto en `data/00_raw/`.

---

In [1]:
# ============================================================================
# CELDA 1 — CONFIGURACIÓN DE RUTAS (ROOT robusto)
# ============================================================================
# Detecta automáticamente la raíz del proyecto subiendo niveles hasta
# encontrar la carpeta src/. Idéntico patrón al resto de notebooks F0.
# ============================================================================

import sys
import warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')

def _encontrar_root(start: Path) -> Path:
    """Sube por los padres hasta encontrar la carpeta src/."""
    for parent in [start] + list(start.parents):
        if (parent / 'src').is_dir():
            return parent
    raise FileNotFoundError(f'No se encontró src/ subiendo desde {start}')

ROOT = _encontrar_root(Path.cwd())
sys.path.insert(0, str(ROOT))

# --- Imports del proyecto ---
from src.config import EXCEL_PRINCIPAL, EXCEL_PREINSCRIPCION, RUTA_RAW

# Carpeta donde está el zip DEMO en el repositorio
RUTA_DEMO = ROOT / 'data' / 'demo'
ZIP_DEMO  = RUTA_DEMO / 'demo_excel.zip'

# Constantes del notebook
UMBRAL_FILAS_ORIGINAL = 1000   # >1000 filas → original, NO machacar
UMBRAL_FILAS_DEMO     = 100    # <100 filas  → DEMO antiguo, renombrar a backup

# Nombre del fichero protegido que NUNCA se toca
FICHERO_PROTEGIDO = 'datos_proyecto_sin_preinscrip.ORIGINALxlsx'

print(f'📂 ROOT del proyecto: {ROOT}')
print(f'📦 ZIP DEMO esperado: {ZIP_DEMO}')
print(f'📁 Carpeta destino:   {RUTA_RAW}')

📂 ROOT del proyecto: c:\FF\AU_UJI_v2
📦 ZIP DEMO esperado: c:\FF\AU_UJI_v2\data\demo\demo_excel.zip
📁 Carpeta destino:   c:\FF\AU_UJI_v2\data\00_raw


In [2]:
# ============================================================================
# CELDA 2 — VERIFICACIONES PREVIAS
# ============================================================================
# 1. Comprobar que existe el zip DEMO (descargado del repo).
# 2. Detectar carpeta antigua data/00_raw/ejemplo/ (avisar, NO borrar).
# ============================================================================

print('=' * 60)
print('VERIFICACIONES PREVIAS')
print('=' * 60)

# --- 1. ZIP DEMO ---
if not ZIP_DEMO.exists():
    raise FileNotFoundError(
        f'❌ No se encuentra el ZIP DEMO: {ZIP_DEMO}\n'
        f'   Asegúrate de haber descargado el repositorio completo desde GitHub.\n'
        f'   El fichero debería estar en: data/demo/demo_excel.zip'
    )

tam_zip_kb = ZIP_DEMO.stat().st_size / 1024
print(f'\n   ✅ ZIP DEMO encontrado: {tam_zip_kb:,.1f} KB')

# --- 2. Carpeta antigua data/00_raw/ejemplo/ ---
carpeta_antigua = RUTA_RAW / 'ejemplo'
if carpeta_antigua.exists():
    ficheros_antiguos = list(carpeta_antigua.glob('*'))
    print(f'\n   ⚠️  Detectada carpeta antigua: {carpeta_antigua}')
    print(f'      Contiene {len(ficheros_antiguos)} fichero(s):')
    for f in ficheros_antiguos:
        tam = f.stat().st_size / 1024 if f.is_file() else 0
        print(f'         • {f.name} ({tam:,.1f} KB)')
    print(f'      → Esta carpeta es de una versión anterior y ya NO se usa.')
    print(f'      → Puedes borrarla manualmente cuando quieras (no es necesaria).')
else:
    print(f'\n   ✅ No hay carpeta antigua data/00_raw/ejemplo/ (correcto)')

VERIFICACIONES PREVIAS


FileNotFoundError: ❌ No se encuentra el ZIP DEMO: c:\FF\AU_UJI_v2\data\demo\demo_excel.zip
   Asegúrate de haber descargado el repositorio completo desde GitHub.
   El fichero debería estar en: data/demo/demo_excel.zip

In [ ]:
# ============================================================================
# CELDA 3 — DETECCIÓN AUTOMÁTICA: ¿hay datos originales en data/00_raw/?
# ============================================================================
# Si encuentra ficheros con los nombres reales, los analiza:
#   - >1.000 filas → ORIGINALES → NO machacar (avisar y salir).
#   - <100 filas  → DEMO antiguos → renombrar a _BACKUP_<fecha>.xlsx.
#   - entre 100 y 1.000 → caso ambiguo → renombrar y avisar.
# ============================================================================

import pandas as pd

print('=' * 60)
print('DETECCIÓN DE FICHEROS EXISTENTES EN data/00_raw/')
print('=' * 60)

es_original_existente = False
fichero_a_revisar = EXCEL_PRINCIPAL

if fichero_a_revisar.exists():
    print(f'\n📖 Encontrado: {fichero_a_revisar.name}')
    try:
        df_check = pd.read_excel(fichero_a_revisar, sheet_name='Expedientes')
        n_filas = len(df_check)
        print(f'   Filas en hoja Expedientes: {n_filas:,}')

        if n_filas > UMBRAL_FILAS_ORIGINAL:
            es_original_existente = True
            print(f'\n   🛡️  PROTECCIÓN ACTIVADA — más de {UMBRAL_FILAS_ORIGINAL:,} filas detectadas.')
            print(f'       Estos son tus DATOS REALES, NO los machaco.')
        else:
            print(f'\n   ℹ️  Menos de {UMBRAL_FILAS_ORIGINAL:,} filas → parece un DEMO previo.')
            print(f'       Se renombrará como backup antes de copiar el nuevo.')
    except Exception as e:
        print(f'   ⚠️  No se pudo leer la hoja Expedientes: {e}')
        print(f'   → Tratamos el fichero como sospechoso, lo renombraremos como backup.')
else:
    print('\n   ✅ No hay ficheros previos en data/00_raw/ con esos nombres')

# Verificar también el fichero protegido (NUNCA tocarlo, solo informar)
fichero_protegido_path = RUTA_RAW / FICHERO_PROTEGIDO
if fichero_protegido_path.exists():
    print(f'\n   🔒 Detectado: {FICHERO_PROTEGIDO}')
    print(f'      Este es el dataset ORIGINAL completo (toda la Comunitat Valenciana).')
    print(f'      NUNCA se toca. Permanece intacto en su sitio.')

# Salir si los originales están
if es_original_existente:
    print('\n' + '=' * 60)
    print('🛑 EJECUCIÓN DETENIDA')
    print('=' * 60)
    print('Tienes los datos originales en data/00_raw/. NO se han modificado.')
    print('Si quieres usar los DEMO en su lugar:')
    print('  1. Mueve manualmente los Excel originales a otra carpeta.')
    print('  2. Vuelve a ejecutar este notebook.')
    raise SystemExit('Datos originales protegidos — no se realizan cambios.')

In [ ]:
# ============================================================================
# CELDA 4 — RENOMBRAR DEMO ANTIGUOS A BACKUP (si existen)
# ============================================================================
# Para los ficheros con nombres reales que ya estaban (DEMO antiguos),
# se renombran añadiendo un sufijo _BACKUP_YYYYMMDD_HHMMSS.
# Esto es REVERSIBLE: el usuario puede recuperarlos quitando el sufijo.
# ============================================================================

print('=' * 60)
print('BACKUP DE DEMO ANTIGUOS (si existen)')
print('=' * 60)

ficheros_a_proteger = [EXCEL_PRINCIPAL, EXCEL_PREINSCRIPCION]
fecha_hora = datetime.now().strftime('%Y%m%d_%H%M%S')

n_renombrados = 0
for fichero in ficheros_a_proteger:
    if fichero.exists():
        nuevo_nombre = fichero.with_name(f'{fichero.stem}_BACKUP_{fecha_hora}{fichero.suffix}')
        fichero.rename(nuevo_nombre)
        print(f'   ✅ Renombrado: {fichero.name}')
        print(f'              → {nuevo_nombre.name}')
        n_renombrados += 1

if n_renombrados == 0:
    print('\n   ✅ No había ficheros previos que renombrar')
else:
    print(f'\n   📌 {n_renombrados} fichero(s) renombrado(s) con sufijo _BACKUP_{fecha_hora}')

In [ ]:
# ============================================================================
# CELDA 5 — DESCOMPRIMIR EL ZIP DEMO EN data/00_raw/
# ============================================================================
# El zip contiene los 2 Excel directamente (sin subcarpetas).
# Se extraen a data/00_raw/ con sus nombres reales.
# ============================================================================

import zipfile

print('=' * 60)
print('INSTALANDO DEMO EN data/00_raw/')
print('=' * 60)

RUTA_RAW.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_DEMO, 'r') as zf:
    print(f'\n📦 Contenido del ZIP:')
    for nombre in zf.namelist():
        info = zf.getinfo(nombre)
        print(f'   • {nombre} ({info.file_size / 1024:,.1f} KB)')

    # Extraer todo a data/00_raw/
    zf.extractall(RUTA_RAW)

print(f'\n✅ Extracción completada en: {RUTA_RAW}')

In [ ]:
# ============================================================================
# CELDA 6 — VERIFICACIÓN FINAL
# ============================================================================
# Comprobar que los 2 Excel quedan en data/00_raw/ con los nombres correctos.
# ============================================================================

print('=' * 60)
print('VERIFICACIÓN FINAL')
print('=' * 60)

todo_ok = True

for fichero in [EXCEL_PRINCIPAL, EXCEL_PREINSCRIPCION]:
    if fichero.exists():
        tam_kb = fichero.stat().st_size / 1024
        print(f'   ✅ {fichero.name}: {tam_kb:,.1f} KB')
    else:
        print(f'   ❌ FALTA: {fichero.name}')
        todo_ok = False

print()
if todo_ok:
    print('=' * 60)
    print('✅ DEMO INSTALADOS CORRECTAMENTE')
    print('=' * 60)
    print()
    print('📌 Siguientes pasos:')
    print('   1. Ejecutar f0_validar_excel.ipynb para validar la estructura.')
    print('   2. Ejecutar orquestador_maestro.ipynb para lanzar el pipeline.')
else:
    print('⚠️ Algo no ha ido bien. Revisar los mensajes anteriores.')